In [ ]:
import glob, io, os, numpy as np, torch
from PIL import Image
import torch.nn.functional as F

test_dir = os.environ.get('TEST_DIR') or sorted(glob.glob('/kaggle/input/**/test_images', recursive=True))[0]
files = sorted(glob.glob(test_dir + '/*'))
print('hidden images:', len(files), '| first:', os.path.basename(files[0]))
raw = [Image.open(f) for f in files]
print('modes:', {m: sum(i.mode == m for i in raw) for m in {i.mode for i in raw}}, '| sizes:', {i.size for i in raw})
hidden = np.stack([np.asarray(i.convert('RGB')) for i in raw])

def stats(x):
    t = torch.from_numpy(x).float().permute(0, 3, 1, 2) / 255
    g = t.mean(1, keepdim=True)
    lap = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]]).float()[None, None]
    sharp = F.conv2d(g, lap).flatten(1).var(1)
    f = torch.fft.fftshift(torch.fft.fft2(g[:, 0]).abs() ** 2, dim=(1, 2))
    yy, xx = torch.meshgrid(torch.arange(32) - 16, torch.arange(32) - 16, indexing='ij')
    hi = f[:, (yy ** 2 + xx ** 2).sqrt() > 11].sum(1) / f.flatten(1).sum(1)
    d = (g[..., 1:] - g[..., :-1]).abs().mean((1, 2))
    edge = d[:, 7::8].mean(1) / d[:, [i for i in range(31) if i % 8 != 7]].mean(1)
    uniq = np.mean([len(np.unique(i.reshape(-1, 3), axis=0)) / 1024 for i in x])
    return dict(R=t[:, 0].mean(), G=t[:, 1].mean(), B=t[:, 2].mean(), bright_sd=g.mean((1, 2, 3)).std(),
                contrast=g.flatten(1).std(1).mean(), sharp=1e3 * sharp.median(), hifreq=1e4 * hi.median(),
                block8=edge.mean(), colors=uniq, saturated=100 * ((t < .01) | (t > .99)).float().mean())

REF = {'cifar test': {'R': 0.497, 'G': 0.489, 'B': 0.455, 'bright_sd': 0.126, 'contrast': 0.194, 'sharp': 31.948, 'hifreq': 42.005, 'block8': 1.038, 'colors': 0.897, 'saturated': 2.351}, 'cifar train': {'R': 0.491, 'G': 0.484, 'B': 0.447, 'bright_sd': 0.123, 'contrast': 0.195, 'sharp': 33.577, 'hifreq': 43.113, 'block8': 1.044, 'colors': 0.902, 'saturated': 2.331}, 'test jpeg90': {'R': 0.497, 'G': 0.489, 'B': 0.455, 'bright_sd': 0.126, 'contrast': 0.194, 'sharp': 33.825, 'hifreq': 44.706, 'block8': 1.052, 'colors': 0.887, 'saturated': 2.359}, 'test jpeg75': {'R': 0.497, 'G': 0.489, 'B': 0.455, 'bright_sd': 0.126, 'contrast': 0.194, 'sharp': 31.863, 'hifreq': 42.163, 'block8': 1.145, 'colors': 0.886, 'saturated': 2.492}}

S = {'HIDDEN': {k: float(v) for k, v in stats(hidden).items()}, **REF}
keys = list(S['HIDDEN'])
print(f"{'':13}" + ''.join(f'{k:>10}' for k in keys))
for k, v in S.items():
    print(f'{k:13}' + ''.join(f'{v[c]:10.3f}' for c in keys))

grid = np.concatenate([np.concatenate(list(hidden[i * 12:(i + 1) * 12]), 1) for i in range(8)], 0)
Image.fromarray(grid).resize((grid.shape[1] * 5, grid.shape[0] * 5), Image.NEAREST).save('hidden_grid.png')
display(Image.open('hidden_grid.png')) if 'display' in globals() else None
